In [ ]:
# Cell 1 - Bronze Layer
from pyspark.sql.functions import current_timestamp, col

# Path to your Unity Catalog Volume
raw_path = "/Volumes/example_data/default/data/retail_demo/raw_orders.csv"

# Read Raw data
bronze_df = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load(raw_path)

# Add metadata for the Bronze audit trail
bronze_with_metadata = bronze_df.withColumn("ingested_at", current_timestamp())

# Save to Unity Catalog (example_data.default.orders_bronze)

bronze_with_metadata.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("example_data.default.orders_bronze")

print("Bronze Layer: Data Ingested from Volume to example_data.default.orders_bronze")

In [ ]:
# Cell 2 - Silver Lyaer (Transformation)

from pyspark.sql.functions import col, when

# Read from the Bronze Delta Table
# Unity catalog automatically optimizes this read
silver_df = spark.table("example_data.default.orders_bronze") \
    .filter(col("amount") > 0) \
    .filter(col("customer_name").isNotNull() & (col("customer_name") != "Null")) \
    .withColumn("is_high_value", col("amount") > 200)

# Save as the Silver Table
silver_df.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("example_data.default.orders_silver")

print("Silver Layer: Data cleaned and saved to example_data.default.orders_silver")

In [ ]:
%sql
--  Cell 3 - Gold Layer
-- We switch to SQL for the Gold layer because it's the standard for aggregation
CREATE OR REPLACE TABLE example_data.default.orders_gold AS
SELECT
  category,
  ROUND(SUM(amount), 2) as total_revenue,
  COUNT(order_id) as total_orders,
  current_date() as report_data
from example_data.default.orders_silver
GROUP BY category
ORDER BY total_revenue DESC;

SELECT * FROM example_data.default.orders_gold;

In [ ]:
# Cell 4 - Final View
# Display the results in the notebook
display(spark.table("orders_gold"))